# BT4012: Student-selected fraud-guided semi-supervised refinement

**11 October 2026.** After the reconstruction-only comparison, the student explicitly chose this fixed fraud-guided refinement: known-label binary cross-entropy plus 0.1 times reconstruction loss, keeping the encoder architecture, original XGBoost settings and four chronological checks. No automatic search, blending, model selection or Kaggle upload is performed.

This is a newly trained model. The previous all-row reconstruction-only model averaged 0.911090 AUC versus the original XGBoost's 0.927761. The best recorded public score remains 0.94265. These prior results motivate testing a different objective; they do not prove the reason for the decline.

## 1. How known and unknown transactions contribute

The encoder learns two tasks. Its decoder reconstructs clean transaction features from slightly noisy inputs, using every permitted feature row. A linear classification head learns licit/illicit answers only for confirmed training labels. Unknown rows contribute reconstruction only; they are never assigned guessed fraud labels.

The joint batch objective is **known-label binary cross-entropy + 0.1 × mean squared reconstruction error**. Binary cross-entropy is averaged over known rows in that batch. Reconstruction error is averaged over all rows and all 165 features. With no known rows, only reconstruction contributes. Positive-class weight is 1. Labels are used only within the permitted fitting period.

The hypothesis is that fraud supervision makes the learned representation more relevant to fraud ranking than reconstruction alone. This is a hypothesis, not a measured causal explanation or guarantee. The classification head guides the encoder; its score is not appended to XGBoost. We retain the same 32 latent values, one reconstruction error, and original 165 features, for 198 XGBoost inputs.

General method references only: [semi-supervised learning](https://scikit-learn.org/stable/modules/semi_supervised.html), [denoising representation learning](https://www.jmlr.org/papers/v11/vincent10a.html). Only supplied competition data is used.

## 2. Student-selected settings and controls

Compare original XGBoost, a fraud-guided encoder trained on known rows, and a fraud-guided encoder trained on all permitted rows. The known-only and all-row conditions use the same joint objective. Scaling uses the respective pretraining rows. The all-row condition has more gradient updates because both conditions use 30 complete passes; the comparison therefore assesses the entire addition of unlabelled data, including scaling and update count.

Encoder: 165→128→32. Decoder: 32→128→165. Linear fraud head: 32→1. ReLU hidden layers; linear latent/reconstruction outputs. StandardScaler fitted on permitted rows, clipping to ±10, Gaussian noise SD 0.1, 30 passes, batch size 512, Adam rate 0.001, weight decay 0.0001, seed 42, two CPU threads. The only conceptual refinement is adding the confirmed-label objective and its 0.1 reconstruction weighting; no selected value is changed after seeing outer scores.

Retain original XGBoost: 15 leaves, 127 bins, rate 0.05, lambda 10, minimum child weight 3, full sampling, positive-class weight 1, seed 42, up to 400 trees and 60-round stopping.

In [1]:
from pathlib import Path
import hashlib
import json
import platform
import subprocess
import sys
import time

import numpy as np
import pandas as pd
import sklearn
import xgboost
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits
from xgboost import XGBClassifier

data_locations = [Path.cwd() / 'Data Set', Path.cwd().parent / 'Data Set',
                  Path('/Users/mlvlo/BT4012/Kaggle/Data Set')]
DATA_DIR = next((p for p in data_locations if (p / 'Train Data.csv').is_file()), None)
assert DATA_DIR is not None, 'Set DATA_DIR to the supplied competition files.'
OUTPUT_DIR = Path.cwd() / 'fraud_guided_autoencoder_results'
OUTPUT_DIR.mkdir(exist_ok=True)
(OUTPUT_DIR / 'encoders').mkdir(exist_ok=True)
FEATURES = [f'feat_{i}' for i in range(1, 166)]
AE_FEATURES = [f'ae_{i:02d}' for i in range(1, 33)] + ['ae_reconstruction_mse']
CHECKS = [(14, 15, 28), (17, 18, 31), (19, 20, 33), (21, 22, 35)]
FAMILIES = ['original_xgboost', 'joint_known_only', 'joint_all_rows']
SEED, CPU_THREADS, EPOCHS, BATCH_SIZE = 42, 2, 30, 512
NOISE_SD, ADAM_RATE, WEIGHT_DECAY = 0.1, 0.001, 0.0001
RECONSTRUCTION_WEIGHT = 0.1
MAXIMUM_ROUNDS, STOPPING_PATIENCE = 400, 60
assert 'torch' not in sys.modules, 'Use a fresh process for this notebook.'

train_path = DATA_DIR / 'Train Data.csv'
train = pd.read_csv(train_path, usecols=['txId', 'time_step', 'label'] + FEATURES)
assert train.txId.is_unique and train.time_step.between(1, 35).all()
assert np.isfinite(train[FEATURES].to_numpy()).all()
known = train.loc[train.label.notna()].copy()
assert known.label.isin([0, 1]).all()
known['label'] = known.label.astype(int)
assert known.label.nunique() == 2
encoder_records = []
worker_call_number = 0
torch_version = None

def progress(**details):
    temporary = OUTPUT_DIR / 'progress.tmp'
    temporary.write_text(json.dumps(details))
    temporary.replace(OUTPUT_DIR / 'progress.json')

print('Training rows:', len(train), '; known:', len(known), '; unknown:', int(train.label.isna().sum()))
print('XGBoost:', xgboost.__version__, '; CPU threads:', CPU_THREADS)


Training rows: 141772 ; known: 31235 ; unknown: 110537
XGBoost: 3.2.0 ; CPU threads: 2


## 3. Fit the fraud-guided encoder in an isolated process

The worker receives standardized features and a label array with missing entries preserved. It computes fraud loss only where the label is confirmed. Missing labels never become zero. Reconstruction uses every permitted feature row. Save both component losses and the joint loss for each epoch. Lower training losses do not establish higher future AUC.

PyTorch and XGBoost run in separate processes to avoid the native OpenMP conflict diagnosed in the preceding experiment. The worker's complete source is embedded below and generated by this notebook. There is no outside script or competition solution dependency. Saved encoder state is checked against its in-memory predictions after fitting.

In [2]:
WORKER_SOURCE = r'''
from pathlib import Path
import json
import sys
import time
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F

configuration = json.loads(sys.argv[1])
torch.set_num_threads(configuration['threads'])
torch.set_num_interop_threads(configuration['threads'])
torch.use_deterministic_algorithms(True)

class FraudGuidedAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(165, 128), nn.ReLU(), nn.Linear(128, 32))
        self.decoder = nn.Sequential(nn.Linear(32, 128), nn.ReLU(), nn.Linear(128, 165))
        self.fraud_head = nn.Linear(32, 1)

    def forward(self, noisy_features):
        return self.decoder(self.encoder(noisy_features))

def encode(clean, model):
    batches = []
    model.eval()
    with torch.no_grad():
        for first in range(0, len(clean), configuration['batch_size']):
            inputs = torch.from_numpy(clean[first:first + configuration['batch_size']])
            latent = model.encoder(inputs)
            error = (inputs - model.decoder(latent)).square().mean(dim=1, keepdim=True)
            batches.append(torch.cat([latent, error], dim=1).numpy())
    return np.concatenate(batches, axis=0)

clean_array = np.load(configuration['input_path'], allow_pickle=False)
assert clean_array.dtype == np.float32 and np.isfinite(clean_array).all()
started = time.perf_counter()
if configuration['action'] == 'fit':
    torch.manual_seed(configuration['seed'])
    model = FraudGuidedAutoencoder()
    optimizer = torch.optim.Adam(model.parameters(), lr=configuration['learning_rate'],
                                 weight_decay=configuration['weight_decay'])
    shuffle_generator = torch.Generator().manual_seed(configuration['seed'])
    clean = torch.from_numpy(clean_array)
    label_array = np.load(configuration['labels_path'], allow_pickle=False)
    assert label_array.shape == (len(clean_array),)
    assert np.isin(label_array[np.isfinite(label_array)], [0, 1]).all()
    labels = torch.from_numpy(label_array)
    curve, reconstruction_curve, classification_curve = [], [], []
    for epoch in range(1, configuration['epochs'] + 1):
        model.train()
        order = torch.randperm(len(clean), generator=shuffle_generator)
        total_loss = 0.0
        total_reconstruction = 0.0
        total_classification = 0.0
        known_count = 0
        for first in range(0, len(clean), configuration['batch_size']):
            indices = order[first:first + configuration['batch_size']]
            batch = clean[indices]
            batch_labels = labels[indices]
            labelled = torch.isfinite(batch_labels)
            noisy = batch + configuration['noise_sd'] * torch.randn_like(batch)
            optimizer.zero_grad()
            latent = model.encoder(noisy)
            reconstruction_loss = (model.decoder(latent) - batch).square().mean()
            if labelled.any():
                logits = model.fraud_head(latent).squeeze(1)
                classification_loss = F.binary_cross_entropy_with_logits(
                    logits[labelled], batch_labels[labelled])
                total_classification += float(classification_loss.detach()) * int(labelled.sum())
                known_count += int(labelled.sum())
            else:
                classification_loss = reconstruction_loss.new_zeros(())
            loss = classification_loss + configuration['reconstruction_weight'] * reconstruction_loss
            assert torch.isfinite(loss), 'Non-finite joint training loss.'
            loss.backward()
            optimizer.step()
            total_loss += float(loss.detach()) * len(batch)
            total_reconstruction += float(reconstruction_loss.detach()) * len(batch)
        assert known_count == int(torch.isfinite(labels).sum())
        curve.append(total_loss / len(clean))
        reconstruction_curve.append(total_reconstruction / len(clean))
        classification_curve.append(total_classification / known_count)
        progress_path = Path(configuration['progress_path'])
        temporary = progress_path.with_suffix('.tmp')
        temporary.write_text(json.dumps(dict(
            phase='joint encoder training', family=configuration['family'],
            cutoff=configuration['end'], stage=configuration['stage'],
            epoch=epoch, epochs=configuration['epochs'], rows=len(clean),
            seconds=round(time.perf_counter() - started, 2))))
        temporary.replace(progress_path)
    assert all(torch.isfinite(parameter).all() for parameter in model.parameters())
    torch.save(model.state_dict(), configuration['weights_path'])
    restored = FraudGuidedAutoencoder()
    restored.load_state_dict(torch.load(configuration['weights_path'], map_location='cpu', weights_only=True))
    probe = clean_array[:min(len(clean_array), 1024)]
    assert np.array_equal(encode(probe, model), encode(probe, restored))
    print(json.dumps(dict(torch_version=torch.__version__, curve=curve, reconstruction_curve=reconstruction_curve,
                          classification_curve=classification_curve,
                          saved_encoder_reload_verified=True)))
else:
    assert configuration['action'] == 'encode'
    model = FraudGuidedAutoencoder()
    model.load_state_dict(torch.load(configuration['weights_path'], map_location='cpu', weights_only=True))
    summaries = encode(clean_array, model)
    assert summaries.shape == (len(clean_array), 33) and np.isfinite(summaries).all()
    np.save(configuration['output_path'], summaries, allow_pickle=False)
    print(json.dumps(dict(torch_version=torch.__version__)))
'''
WORKER_PATH = OUTPUT_DIR / 'encoder_worker.py'
WORKER_PATH.write_text(WORKER_SOURCE)


def run_encoder_worker(clean_array, configuration, labels=None):
    global worker_call_number, torch_version
    worker_call_number += 1
    input_path = OUTPUT_DIR / f'.encoder_input_{worker_call_number}.npy'
    output_path = OUTPUT_DIR / f'.encoder_output_{worker_call_number}.npy'
    labels_path = OUTPUT_DIR / f'.encoder_labels_{worker_call_number}.npy'
    if labels is not None:
        np.save(labels_path, labels, allow_pickle=False)
        configuration = dict(configuration, labels_path=str(labels_path))
    np.save(input_path, clean_array, allow_pickle=False)
    configuration = dict(configuration, input_path=str(input_path), output_path=str(output_path),
                         threads=CPU_THREADS, batch_size=BATCH_SIZE)
    result = subprocess.run([sys.executable, str(WORKER_PATH), json.dumps(configuration)],
                            capture_output=True, text=True, timeout=600)
    assert result.returncode == 0, f'Encoder worker failed: {result.stderr}'
    details = json.loads(result.stdout)
    torch_version = details['torch_version']
    summaries = np.load(output_path, allow_pickle=False) if output_path.exists() else None
    input_path.unlink()
    if labels_path.exists():
        labels_path.unlink()
    if output_path.exists():
        output_path.unlink()
    return details, summaries


def pretraining_rows(family, end):
    assert family in ['joint_known_only', 'joint_all_rows']
    rows = train.loc[train.time_step <= end]
    if family == 'joint_known_only':
        rows = rows.loc[rows.label.notna()]
    assert len(rows) and rows.time_step.max() <= end
    return rows


def fit_encoder(feature_rows, *, family, end, stage):
    raw = feature_rows[FEATURES].to_numpy(dtype=np.float32)
    assert raw.shape[1] == 165 and np.isfinite(raw).all()
    scaler = StandardScaler().fit(raw)
    clean_array = np.clip(scaler.transform(raw), -10, 10).astype(np.float32)
    started = time.perf_counter()
    prefix = f'{family}_{end}_{stage}'
    weights_path = OUTPUT_DIR / 'encoders' / f'{prefix}.pt'
    details, _ = run_encoder_worker(clean_array, dict(
        action='fit', weights_path=str(weights_path), epochs=EPOCHS, seed=SEED,
        noise_sd=NOISE_SD, learning_rate=ADAM_RATE, weight_decay=WEIGHT_DECAY,
        family=family, end=end, stage=stage, reconstruction_weight=RECONSTRUCTION_WEIGHT,
        progress_path=str(OUTPUT_DIR / 'progress.json')), labels=feature_rows.label.to_numpy(dtype=np.float32))
    assert details['saved_encoder_reload_verified']
    curve = details['curve']
    np.savez_compressed(OUTPUT_DIR / 'encoders' / f'{prefix}_scaler.npz',
                        mean=scaler.mean_, scale=scaler.scale_, var=scaler.var_,
                        n_samples_seen=np.asarray(scaler.n_samples_seen_))
    pd.DataFrame({'epoch': np.arange(1, EPOCHS + 1), 'joint_loss': curve,
                  'noisy_reconstruction_mse': details['reconstruction_curve'],
                  'known_label_binary_cross_entropy': details['classification_curve']}).to_csv(
        OUTPUT_DIR / 'encoders' / f'{prefix}_loss.csv', index=False)
    encoder_records.append(dict(family=family, train_end=end, stage=stage,
                                rows=len(feature_rows), known_rows=int(feature_rows.label.notna().sum()),
                                unknown_rows=int(feature_rows.label.isna().sum()),
                                first_epoch_joint_loss=curve[0], last_epoch_joint_loss=curve[-1],
                                first_epoch_mse=details['reconstruction_curve'][0],
                                last_epoch_mse=details['reconstruction_curve'][-1],
                                first_epoch_bce=details['classification_curve'][0],
                                last_epoch_bce=details['classification_curve'][-1],
                                seconds=round(time.perf_counter() - started, 3)))
    pd.DataFrame(encoder_records).to_csv(OUTPUT_DIR / 'encoder_training.csv', index=False)
    return weights_path, scaler


def augmented_features(rows, encoder, scaler):
    clean = np.clip(scaler.transform(rows[FEATURES].to_numpy(dtype=np.float32)), -10, 10).astype(np.float32)
    _, summaries = run_encoder_worker(clean, dict(action='encode', weights_path=str(encoder)))
    assert summaries.shape == (len(rows), 33) and np.isfinite(summaries).all()
    extra = pd.DataFrame(summaries, index=rows.index, columns=AE_FEATURES)
    features = pd.concat([rows[FEATURES], extra], axis=1)
    assert features.shape == (len(rows), 198) and features.index.equals(rows.index)
    assert not any(name in features.columns for name in ['label', 'time_step', 'txId', 'index'])
    return features


## 4. Keep chronological selection and refitting

For cutoff 14, the initial scaler and encoder see only permitted rows through 11. The classification head receives known labels only through 11. XGBoost fits known labels through 11 and stops using known labels in 12–14. After selecting its tree count, refit the encoder within 1–14, using confirmed labels through 14, and fit XGBoost on confirmed labels through 14. Predict 15–28 without fitting on its features or labels. Other cutoffs use the same last-three-period stopping rule.

The classifier-guided encoder is supervised on the same permitted fitting labels later used by XGBoost. This can overfit training representations, so outer chronological checks are essential. It is not allowed to receive outer checking answers. Refit changes the representation; the selected tree count is a carried-over recipe rather than a proven optimum for the refitted representation.

The four checks overlap and are reused development evidence. Original-control predictions must reproduce the earlier arrays when available; earlier arrays are an optional audit and are not required to generate predictions from the competition CSVs.

In [3]:
def make_xgboost(rounds, *, stopping=False):
    return XGBClassifier(
        objective='binary:logistic', n_estimators=int(rounds), learning_rate=0.05,
        tree_method='hist', device='cpu', grow_policy='lossguide', max_depth=0,
        max_leaves=15, max_bin=127, min_child_weight=3, reg_lambda=10,
        subsample=1.0, colsample_bytree=1.0, scale_pos_weight=1.0,
        eval_metric='auc', early_stopping_rounds=STOPPING_PATIENCE if stopping else None,
        n_jobs=CPU_THREADS, random_state=SEED,
    )


def fit_pipeline(family, end):
    learning = known.loc[known.time_step <= end]
    fitting = learning.loc[learning.time_step <= end - 3]
    stopping = learning.loc[learning.time_step > end - 3]
    assert fitting.time_step.max() < stopping.time_step.min()
    assert fitting.label.nunique() == stopping.label.nunique() == 2
    if family == 'original_xgboost':
        fit_features, stop_features = fitting[FEATURES], stopping[FEATURES]
        encoder, scaler = None, None
    else:
        inner_rows = pretraining_rows(family, end - 3)
        assert set(inner_rows.txId).isdisjoint(stopping.txId)
        encoder, scaler = fit_encoder(inner_rows, family=family, end=end - 3, stage='inner')
        fit_features = augmented_features(fitting, encoder, scaler)
        stop_features = augmented_features(stopping, encoder, scaler)
    progress(phase='XGBoost stopping', family=family, cutoff=end)
    with threadpool_limits(limits=CPU_THREADS):
        initial = make_xgboost(MAXIMUM_ROUNDS, stopping=True)
        initial.fit(fit_features, fitting.label,
                    eval_set=[(stop_features, stopping.label)], verbose=False)
        rounds = int(initial.best_iteration + 1)
        assert 1 <= rounds <= MAXIMUM_ROUNDS
        assert abs(roc_auc_score(stopping.label, initial.predict_proba(stop_features)[:, 1])
                   - float(initial.best_score)) < 1e-12
    curve = initial.evals_result()['validation_0']['auc']
    pd.DataFrame({'round': np.arange(1, len(curve) + 1), 'inner_roc_auc': curve}).to_csv(
        OUTPUT_DIR / f'stopping_curve_{family}_{end}.csv', index=False)
    if family == 'original_xgboost':
        refit_features = learning[FEATURES]
        pretraining_known, pretraining_unknown = 0, 0
    else:
        full_rows = pretraining_rows(family, end)
        encoder, scaler = fit_encoder(full_rows, family=family, end=end, stage='refit')
        refit_features = augmented_features(learning, encoder, scaler)
        pretraining_known = int(full_rows.label.notna().sum())
        pretraining_unknown = int(full_rows.label.isna().sum())
    progress(phase='XGBoost refitting', family=family, cutoff=end, rounds=rounds)
    with threadpool_limits(limits=CPU_THREADS):
        refitted = make_xgboost(rounds)
        refitted.fit(refit_features, learning.label)
    return dict(family=family, end=end, model=refitted, encoder=encoder, scaler=scaler,
                rounds=rounds, learning_rows=len(learning),
                pretraining_known=pretraining_known, pretraining_unknown=pretraining_unknown)


def predict_pipeline(pipeline, rows):
    features = (rows[FEATURES] if pipeline['family'] == 'original_xgboost'
                else augmented_features(rows, pipeline['encoder'], pipeline['scaler']))
    with threadpool_limits(limits=CPU_THREADS):
        return pipeline['model'].predict_proba(features)[:, 1]


## 5. Run the selected fraud-guided comparison

All conditions predict the same checking transactions. Save identifiers, labels, periods and probabilities for independent verification. Do not choose a seed, loss weighting, encoder size or ensemble weight after viewing these outer results.

In [4]:
records, saved = [], {}
started_all = time.perf_counter()
for end, first, last in CHECKS:
    checking = known.loc[known.time_step.between(first, last)]
    assert known.loc[known.time_step <= end].time_step.max() < checking.time_step.min()
    assert set(train.loc[train.time_step <= end].txId).isdisjoint(checking.txId)
    saved[f'labels_{end}'] = checking.label.to_numpy()
    saved[f'txIds_{end}'] = checking.txId.to_numpy()
    saved[f'time_steps_{end}'] = checking.time_step.to_numpy()
    for family in FAMILIES:
        started = time.perf_counter()
        pipeline = fit_pipeline(family, end)
        probabilities = predict_pipeline(pipeline, checking)
        assert np.isfinite(probabilities).all() and np.all((probabilities >= 0) & (probabilities <= 1))
        auc = float(roc_auc_score(checking.label, probabilities))
        saved[f'{family}_{end}'] = probabilities
        row = dict(model=family, train_end=end, validation_start=first, validation_end=last,
                   learning_rows=pipeline['learning_rows'], checking_rows=len(checking),
                   pretraining_known=pipeline['pretraining_known'],
                   pretraining_unknown=pipeline['pretraining_unknown'],
                   rounds=pipeline['rounds'], auc=auc,
                   seconds=round(time.perf_counter() - started, 3))
        records.append(row)
        print(f'{family}: learn through {end}; check {first}–{last}; AUC={auc:.6f}; '
              f'trees={pipeline["rounds"]}; unknown pretraining rows={pipeline["pretraining_unknown"]:,}', flush=True)
        pd.DataFrame(records).to_csv(OUTPUT_DIR / 'validation_results.csv', index=False)
        np.savez_compressed(OUTPUT_DIR / 'validation_probabilities.npz', **saved)
    progress(phase='validation cutoff complete', cutoff=end)

results = pd.DataFrame(records)
comparison = results.pivot(index='train_end', columns='model', values='auc')[FAMILIES]
comparison['all_rows_minus_original'] = comparison.joint_all_rows - comparison.original_xgboost
comparison['all_rows_minus_known_only'] = comparison.joint_all_rows - comparison.joint_known_only
summary = comparison[FAMILIES].agg(['mean', 'min']).T
comparison.to_csv(OUTPUT_DIR / 'comparison.csv')
summary.to_csv(OUTPUT_DIR / 'summary.csv')
print('\nFour chronological checks:')
print(comparison.round(6).to_string())
print('\nMean and worst full-block AUC:')
print(summary.round(6).to_string())
print('Comparison seconds:', round(time.perf_counter() - started_all, 2))
display(comparison.round(6))
display(summary.round(6))


original_xgboost: learn through 14; check 15–28; AUC=0.968521; trees=185; unknown pretraining rows=0
joint_known_only: learn through 14; check 15–28; AUC=0.936518; trees=400; unknown pretraining rows=0
joint_all_rows: learn through 14; check 15–28; AUC=0.895774; trees=300; unknown pretraining rows=56,094
original_xgboost: learn through 17; check 18–31; AUC=0.925611; trees=336; unknown pretraining rows=0
joint_known_only: learn through 17; check 18–31; AUC=0.893006; trees=398; unknown pretraining rows=0
joint_all_rows: learn through 17; check 18–31; AUC=0.864509; trees=74; unknown pretraining rows=64,134
original_xgboost: learn through 19; check 20–33; AUC=0.901582; trees=398; unknown pretraining rows=0
joint_known_only: learn through 19; check 20–33; AUC=0.855061; trees=375; unknown pretraining rows=0
joint_all_rows: learn through 19; check 20–33; AUC=0.820502; trees=115; unknown pretraining rows=68,482
original_xgboost: learn through 21; check 22–35; AUC=0.915330; trees=381; unknown p

model,original_xgboost,joint_known_only,joint_all_rows,all_rows_minus_original,all_rows_minus_known_only
train_end,,,,,
14,0.968521,0.936518,0.895774,-0.072747,-0.040744
17,0.925611,0.893006,0.864509,-0.061102,-0.028497
19,0.901582,0.855061,0.820502,-0.081081,-0.034559
21,0.915330,0.837891,0.824048,-0.091282,-0.013843


,mean,min
model,,
original_xgboost,0.927761,0.901582
joint_known_only,0.880619,0.837891
joint_all_rows,0.851208,0.820502


## 6. Inspect individual periods

The fourteen-step pooled AUC remains the main comparison. Per-period AUC explains improvements and declines; it is undefined for a period with only one class. Do not pool duplicated transactions from overlapping checks.

In [5]:
step_records = []
for end, first, last in CHECKS:
    labels, steps = saved[f'labels_{end}'], saved[f'time_steps_{end}']
    for step in np.unique(steps):
        selected = steps == step
        row = dict(train_end=end, validation_step=int(step), known_labels=int(selected.sum()),
                   illicit=int(labels[selected].sum()))
        for family in FAMILIES:
            row[family] = (float(roc_auc_score(labels[selected], saved[f'{family}_{end}'][selected]))
                           if len(np.unique(labels[selected])) == 2 else np.nan)
        row['all_rows_minus_original'] = row['joint_all_rows'] - row['original_xgboost']
        step_records.append(row)
per_step = pd.DataFrame(step_records)
per_step.to_csv(OUTPUT_DIR / 'per_time_step.csv', index=False)
display(per_step.loc[per_step.train_end == 21].round(6))


,train_end,validation_step,known_labels,illicit,original_xgboost,joint_known_only,joint_all_rows,all_rows_minus_original
42,21,22,1763,158,0.998833,0.995556,0.996072,-0.002760
43,21,23,1187,53,0.957622,0.900702,0.894588,-0.063035
44,21,24,1126,137,0.795488,0.612858,0.558593,-0.236894
45,21,25,594,118,0.999484,0.989158,0.988659,-0.010825
46,21,26,517,96,0.946779,0.895858,0.889573,-0.057205
47,21,27,206,24,0.999542,0.997711,0.997253,-0.002289
48,21,28,284,85,1.000000,0.999645,0.999764,-0.000236
49,21,29,1174,329,0.775619,0.633456,0.595496,-0.180123
50,21,30,524,83,0.987323,0.981805,0.983389,-0.003934
51,21,31,710,106,0.772820,0.693146,0.702049,-0.070770


## 7. Fit full-data experimental conditions

Fit both fixed conditions through step 35. The initial encoder's classification and reconstruction learning end at 32, and known labels in 33–35 choose XGBoost tree count. Refit the all-row encoder on all 141,772 training transactions, using the 31,235 confirmed labels for the fraud objective and all rows for reconstruction. The known-only condition excludes unknown rows from both scaling and reconstruction. Fit XGBoost on all confirmed training labels.

Test features are transformed only after fitting. No test features or answers train the scaler, encoder, classification head or XGBoost. Preparing two checked experimental CSVs does not select either condition for submission. Verify saved components and identifier alignment.

In [6]:
test = pd.read_csv(DATA_DIR / 'Test Data.csv')
template = pd.read_csv(DATA_DIR / 'Sample Submission' / 'BT4012 Competition Submission.csv')
assert test.txId.is_unique and test['index'].is_unique and template['index'].is_unique
assert len(test) == len(template) and set(test['index']) == set(template['index'])
assert known.time_step.max() < test.time_step.min()
assert set(train.txId).isdisjoint(test.txId)
assert np.isfinite(test[FEATURES].to_numpy()).all()
final_records, test_predictions = [], {}
for family in ['joint_known_only', 'joint_all_rows']:
    pipeline = fit_pipeline(family, 35)
    probabilities = predict_pipeline(pipeline, test)
    assert np.isfinite(probabilities).all() and np.all((probabilities >= 0) & (probabilities <= 1))
    mapping = pd.Series(probabilities, index=test['index'])
    submission = template[['index']].copy()
    submission['target'] = submission['index'].map(mapping)
    assert submission.columns.tolist() == ['index', 'target']
    assert submission['index'].equals(template['index']) and submission.target.notna().all()
    submission_path = OUTPUT_DIR / f'submission_{family}_experimental.csv'
    model_path = OUTPUT_DIR / f'xgboost_{family}_experimental.json'
    submission.to_csv(submission_path, index=False, float_format='%.17g')
    pipeline['model'].save_model(model_path)

    # The independent worker reloads the saved encoder for each prediction and
    # also verified its saved state against its in-memory state after fitting.
    restored_encoder = OUTPUT_DIR / 'encoders' / f'{family}_35_refit.pt'
    with np.load(OUTPUT_DIR / 'encoders' / f'{family}_35_refit_scaler.npz') as stored:
        restored_scaler = StandardScaler()
        restored_scaler.mean_ = stored['mean']
        restored_scaler.scale_ = stored['scale']
        restored_scaler.var_ = stored['var']
        restored_scaler.n_samples_seen_ = stored['n_samples_seen']
        restored_scaler.n_features_in_ = len(FEATURES)
    restored_classifier = XGBClassifier(n_jobs=CPU_THREADS)
    restored_classifier.load_model(model_path)
    restored = dict(pipeline, encoder=restored_encoder, scaler=restored_scaler, model=restored_classifier)
    assert np.array_equal(predict_pipeline(restored, test), probabilities)
    csv = pd.read_csv(submission_path)
    assert csv['index'].equals(template['index'])
    assert np.allclose(csv.target, submission.target, rtol=0, atol=1e-12)
    test_predictions[family] = probabilities
    final_records.append(dict(family=family, rounds=pipeline['rounds'],
                              supervised_rows=pipeline['learning_rows'],
                              encoder_known_rows=pipeline['pretraining_known'],
                              encoder_unknown_rows=pipeline['pretraining_unknown'],
                              test_rows=len(test), submission_path=str(submission_path)))
    print('Full-data candidate:', family, '; trees:', pipeline['rounds'],
          '; unknown encoder rows:', pipeline['pretraining_unknown'],
          '; reloaded model and CSV checks passed.', flush=True)
np.savez_compressed(OUTPUT_DIR / 'test_probabilities.npz',
                    txIds=test.txId.to_numpy(), indices=test['index'].to_numpy(), **test_predictions)
pd.DataFrame(final_records).to_csv(OUTPUT_DIR / 'final_training.csv', index=False)
display(pd.DataFrame(final_records))


Full-data candidate: joint_known_only ; trees: 399 ; unknown encoder rows: 0 ; reloaded model and CSV checks passed.
Full-data candidate: joint_all_rows ; trees: 103 ; unknown encoder rows: 110537 ; reloaded model and CSV checks passed.


,family,rounds,supervised_rows,encoder_known_rows,encoder_unknown_rows,test_rows,submission_path
0,joint_known_only,399,31235,31235,0,15329,/Users/mlvlo/.codex/.chatgpt-projects/g-p-6a90...
1,joint_all_rows,103,31235,31235,110537,15329,/Users/mlvlo/.codex/.chatgpt-projects/g-p-6a90...


## 8. Verify and record decisions

The student explicitly chose the fraud-guided objective, reconstruction weight 0.1, unchanged settings and existing chronological checks. Codex implemented and executed that fixed comparison and checked calculations and saved components. No automatic final-model selection or Kaggle upload occurs. Further material refinements remain student decisions.

Repeatedly choosing settings from the same outer periods can overfit development evidence. This experiment cannot establish a hidden public score above 0.94265 or guarantee 0.96+ AUC.

In [7]:
for end, first, last in CHECKS:
    expected = known.loc[known.time_step.between(first, last)]
    assert np.array_equal(saved[f'txIds_{end}'], expected.txId.to_numpy())
    assert np.array_equal(saved[f'labels_{end}'], expected.label.to_numpy())
    for family in FAMILIES:
        assert abs(roc_auc_score(saved[f'labels_{end}'], saved[f'{family}_{end}'])
                   - comparison.loc[end, family]) < 1e-12
previous_path = Path.cwd() / 'feature_ensemble_results' / 'validation_probabilities.npz'
historical_controls_checked = previous_path.is_file()
if historical_controls_checked:
    with np.load(previous_path) as previous:
        for end, first, last in CHECKS:
            assert np.array_equal(saved[f'labels_{end}'], previous[f'labels_{end}'])
            assert np.array_equal(saved[f'original_xgboost_{end}'], previous[f'original_xgboost_{end}'])
# Historical arrays are an optional project audit, not an input needed to
# reproduce predictions from the supplied competition CSVs and this code.

def file_hash(path):
    hasher = hashlib.sha256()
    with path.open('rb') as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b''):
            hasher.update(chunk)
    return hasher.hexdigest()

encoder_config = dict(encoder_widths=[165, 128, 32], decoder_widths=[32, 128, 165],
                      hidden_activation='ReLU', latent_and_output_activation='linear',
                      preprocessing='StandardScaler fit on respective pretraining rows; clip [-10,10]',
                      epochs=EPOCHS, batch_size=BATCH_SIZE, noise_sd=NOISE_SD,
                      optimizer='Adam', learning_rate=ADAM_RATE, weight_decay=WEIGHT_DECAY,
                      seed=SEED, cpu_threads=CPU_THREADS, features_added=len(AE_FEATURES),
                      reconstruction_weight=RECONSTRUCTION_WEIGHT,
                      supervised_objective='binary cross-entropy on confirmed fitting labels only',
                      supervised_class_weight=1, classification_head='linear 32 to 1')
metadata = dict(
    date='2026-10-11', student_decision='train the fixed fraud-guided autoencoder + XGBoost comparison',
    method='joint known-label classification and all-row reconstruction, followed by XGBoost',
    input_rows=len(train), known_labels=len(known), unknown_rows=int(train.label.isna().sum()),
    features=FEATURES, representation_features=AE_FEATURES, encoder_config=encoder_config,
    checks=CHECKS, final_training=final_records, summary=summary.to_dict(orient='index'),
    all_rows_mean_change=float(comparison.all_rows_minus_original.mean()),
    all_rows_mean_change_vs_known_only=float(comparison.all_rows_minus_known_only.mean()),
    original_controls_exactly_reproduced=historical_controls_checked, all_saved_components_reload_exactly=True,
    no_outer_features_or_labels_in_fitting=True, no_test_features_in_pretraining=True,
    outer_checks_overlap=True, outer_checks_are_development_evidence=True,
    equal_encoder_epochs_with_unequal_gradient_update_counts=True,
    original_public_auc=0.94265, candidate_public_auc=None,
    student_final_selection='pending review', kaggle_submission_made=False,
    python=platform.python_version(), torch=torch_version, sklearn=sklearn.__version__,
    runtime='isolated PyTorch subprocess; XGBoost in the notebook process',
    initial_openmp_runtime_crash_corrected=True,
    xgboost=xgboost.__version__, pandas=pd.__version__, numpy=np.__version__,
    input_sha256={name:file_hash(DATA_DIR/name) for name in [
        'Train Data.csv', 'Test Data.csv', 'Sample Submission/BT4012 Competition Submission.csv']},
    submission_sha256={family:file_hash(OUTPUT_DIR/f'submission_{family}_experimental.csv')
                       for family in ['joint_known_only','joint_all_rows']},
)
(OUTPUT_DIR / 'experiment_metadata.json').write_text(json.dumps(metadata, indent=2))
progress(phase='complete', all_rows_mean_auc=float(summary.loc['joint_all_rows','mean']),
         original_mean_auc=float(summary.loc['original_xgboost','mean']))
print('Chronology, probability alignment, AUC, historical controls, reloaded components and CSV checks passed.')
print('No Kaggle upload or automatic final-model selection was made.')


Chronology, probability alignment, AUC, historical controls, reloaded components and CSV checks passed.
No Kaggle upload or automatic final-model selection was made.


## 9. Measured result and diagnosis

All seven code cells executed in sequence and passed. The original-control probabilities exactly reproduced the earlier experiment. Identifiers, AUC recalculation, saved components and CSV alignment passed. No model setting was changed after observing outer results.

| Validation steps | Original XGBoost | Guided known-only encoder | Guided all-row encoder | All-row change |
|---|---:|---:|---:|---:|
| 15–28 | 0.968521 | 0.936518 | 0.895774 | -0.072747 |
| 18–31 | 0.925611 | 0.893006 | 0.864509 | -0.061102 |
| 20–33 | 0.901582 | 0.855061 | 0.820502 | -0.081081 |
| 22–35 | 0.915330 | 0.837891 | 0.824048 | -0.091282 |
| **Mean** | **0.927761** | **0.880619** | **0.851208** | **-0.076553** |

The guided all-row model improved **0 of four** checks against original XGBoost. Its mean change was **-0.076553 AUC**. Its mean change against the guided known-only condition was **-0.029411**. The earlier reconstruction-only all-row model averaged **0.911090**; adding the selected classification objective and loss weighting did not improve that mean.

**AI-assisted recommendation:** Do not spend a remaining submission on this fraud-guided all-row candidate; it declined on every chronological check. The original model remains the best recorded public result, **0.94265**. These new models' public/private scores are unknown. No Kaggle upload was made.

### What the fitted classification head tells us

A separate diagnostic evaluated the already-trained classification heads without refitting, tuning or blending. The guided all-row heads gave:

| Encoder cutoff | In-sample head AUC | Future head AUC |
|---|---:|---:|
| 14 | 0.999977 | 0.924604 |
| 17 | 0.999963 | 0.880019 |
| 19 | 0.999973 | 0.859792 |
| 21 | 0.999847 | 0.862683 |

The heads closely fit their earlier confirmed training answers but ranked later transactions substantially less well. Their mean future head AUC was **0.881774**, below the original XGBoost mean. This is consistent with overfitting and/or differences between periods; this diagnostic alone does not identify the cause. It also does not support choosing the head as a better final predictor. A high in-sample score is not evidence of a high future Kaggle score.

Diagnostic source is saved as `fraud_guided_autoencoder_results/diagnose_fraud_guided_heads.py`; it uses the fitted artifacts and supplied data. This analysis was read-only with respect to model parameters.

### Full-data artifacts and decision ownership

The full guided all-row encoder used **141,772 transactions**, including **all 110,537 unknown training transactions** for reconstruction. Its fraud objective used only the **31,235 confirmed labels**, and its XGBoost classifier used those same permitted known labels. Its final classifier has **103 trees**. Both experimental CSVs contain **15,329 checked test predictions**. No test features or answers fit the encoder, scaler or classifier. Saved components reproduced identical predictions after reload.

The student explicitly selected the known-label classification objective, reconstruction weight 0.1, class weight 1, same network and training settings, and the existing four chronological checks. OpenAI Codex implemented and ran this fixed refinement, used isolated neural/tree processes, diagnosed the fitted heads, verified artifacts and drafted this factual record. The student is responsible for interpretation, subsequent material model choices and final submission selection, and must accurately disclose assistance.

Further material refinements have not been selected or run. Reusing these overlapping development checks for many choices risks overfitting them. No claim of 0.96+ or improvement beyond the recorded public high score has been established.
